# Notebook 04 — Pipeline de Integração Final

## Objetivo

Unir os dois modelos treinados (Notebook 02: categoria de transação;
Notebook 03: perfil financeiro) em um único pipeline de inferência,
replicando exatamente o contrato de entrada/saída definido no exemplo do
edital do hackathon (`POST /analise-financeira`).

## Arquitetura do pipeline

Entrada (JSON do edital)
↓
Para cada transação → Modelo de Categoria → categoria prevista
↓
Soma os valores por categoria → resumo_gastos
↓
Calcula comprometimento_gastos = soma total ÷ renda_mensal
↓
Monta as features → Modelo de Perfil → perfil_financeiro + probabilidade
↓
Aplica regras de recomendação → recomendacoes
↓
Saída (JSON final, igual ao exemplo do edital)

## Nota sobre o teste de aceitação

O teste final deste notebook usa **exatamente** o exemplo de entrada do
edital, incluindo a transação `"Combustivel"` — termo que, numa versão
anterior do vocabulário de estabelecimentos (Notebook 01), não era
reconhecido corretamente e caía na categoria errada (Alimentação em vez
de Transporte). Esse erro real, descoberto durante o teste do pipeline
completo, motivou a correção do vocabulário documentada em
`documentacao_tecnica.md` (Fase 7) — o resultado abaixo já reflete essa
correção.

## Sobre a arquitetura de integração com o Back-end

Este pipeline é o artefato de referência para a integração com a API do
time de Back-end. Duas abordagens de integração foram documentadas em
arquivos separados, dependendo da linguagem escolhida pelo time:
- `CONTRATO_INTEGRACAO_BACKEND.md` — cenário Java/Spring Boot, usando
  OCI Functions como camada intermediária de inferência;
- `CONTRATO_INTEGRACAO_BACKEND_PYTHON.md` — cenário Python/FastAPI,
  carregando os modelos diretamente, com OCI Object Storage + OCI
  Compute.

In [1]:
# ============================================================
# Pipeline de Integração Final — Célula única
# ============================================================
"""
Carrega os artefatos de produção dos dois modelos e executa o pipeline
completo de análise financeira, testando com o exemplo exato do edital
do hackathon.
"""

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
import json
import numpy as np
import pandas as pd
import joblib


def obter_caminhos_laboratorio() -> dict:
    """Centraliza os caminhos de pastas da versão revisada (laboratório)."""
    base = '/content/drive/MyDrive/Hackathon_OCI_G9/HACKATHON G9 - TEAM 20/versao_revisada_final'
    return {'base': base, 'datasets': os.path.join(base, 'datasets'), 'modelos': os.path.join(base, 'modelos')}


def carregar_artefatos_producao(pasta_modelos: str) -> dict:
    """Carrega todos os artefatos de produção necessários para o pipeline."""
    return {
        'vetorizador_tfidf': joblib.load(os.path.join(pasta_modelos, 'vetorizador_tfidf.pkl')),
        'modelo_categoria': joblib.load(os.path.join(pasta_modelos, 'modelo_categoria_producao.pkl')),
        'codificador_categorias': joblib.load(os.path.join(pasta_modelos, 'codificador_categorias.pkl')),
        'modelo_perfil': joblib.load(os.path.join(pasta_modelos, 'modelo_perfil_producao.pkl')),
        'codificador_perfil': joblib.load(os.path.join(pasta_modelos, 'codificador_perfil.pkl')),
    }


def classificar_categorias_transacoes(transacoes: list, artefatos: dict) -> list:
    """Prevê a categoria de cada transação usando o modelo de categoria."""
    descricoes = [t['descricao'] for t in transacoes]
    vetores = artefatos['vetorizador_tfidf'].transform(descricoes)
    categorias_cod = artefatos['modelo_categoria'].predict(vetores)
    categorias = artefatos['codificador_categorias'].inverse_transform(categorias_cod)
    return [{**t, 'categoria': cat} for t, cat in zip(transacoes, categorias)]


def calcular_resumo_gastos(transacoes_classificadas: list) -> dict:
    """Agrupa o valor total gasto por categoria, garantindo todas as 7 categorias no resultado."""
    categorias_possiveis = ['Alimentacao', 'Moradia', 'Transporte', 'Saude', 'Educacao', 'Lazer', 'Servicos']
    resumo = {cat: 0.0 for cat in categorias_possiveis}
    for t in transacoes_classificadas:
        resumo[t['categoria']] += t['valor']
    return {cat: round(valor, 2) for cat, valor in resumo.items()}


def prever_perfil_financeiro(renda_mensal, nivel_endividamento, frequencia_poupanca, resumo_gastos, artefatos) -> tuple:
    """Prevê o perfil financeiro do usuário usando o modelo de perfil."""
    mapa_poupanca = {'Baixa': 0, 'Media': 1, 'Alta': 2}
    comprometimento_gastos = (sum(resumo_gastos.values()) / renda_mensal) * 100

    features = pd.DataFrame([{
        'renda_mensal': renda_mensal, 'nivel_endividamento': nivel_endividamento,
        'frequencia_poupanca_cod': mapa_poupanca[frequencia_poupanca],
        'comprometimento_gastos': comprometimento_gastos, **resumo_gastos,
    }])

    probabilidades = artefatos['modelo_perfil'].predict_proba(features)[0]
    indice = np.argmax(probabilidades)
    perfil = artefatos['codificador_perfil'].classes_[indice]
    return perfil, round(float(probabilidades[indice]), 2)


def gerar_recomendacoes(perfil: str, resumo_gastos: dict, frequencia_poupanca: str) -> list:
    """Gera recomendações simples e objetivas, baseadas em regras de negócio."""
    categoria_maior_gasto = max(resumo_gastos, key=resumo_gastos.get)
    if perfil == 'Em risco':
        return [
            f"Reduzir gastos com {categoria_maior_gasto.lower()}, categoria de maior peso no orçamento",
            "Buscar renegociação de dívidas para reduzir o nível de endividamento",
        ]
    if perfil == 'Em observacao':
        recs = [f"Monitorar gastos recorrentes de {categoria_maior_gasto.lower()}"]
        if frequencia_poupanca == 'Baixa':
            recs.append("Aumentar a frequência de poupança mensal")
        return recs
    return [
        "Manter o padrão atual de organização financeira",
        "Considerar investir o excedente mensal para objetivos de longo prazo",
    ]


def analisar_financas(dados_entrada: dict, artefatos: dict) -> dict:
    """Função principal: recebe os dados brutos e devolve a análise financeira completa."""
    transacoes_classificadas = classificar_categorias_transacoes(dados_entrada['transacoes'], artefatos)
    resumo_gastos = calcular_resumo_gastos(transacoes_classificadas)
    perfil, probabilidade = prever_perfil_financeiro(
        dados_entrada['renda_mensal'], dados_entrada['nivel_endividamento'],
        dados_entrada['frequencia_poupanca'], resumo_gastos, artefatos
    )
    recomendacoes = gerar_recomendacoes(perfil, resumo_gastos, dados_entrada['frequencia_poupanca'])
    return {
        'perfil_financeiro': perfil,
        'probabilidade': probabilidade,
        'resumo_gastos': {k: v for k, v in resumo_gastos.items() if v > 0},
        'recomendacoes': recomendacoes,
    }


# ------------------------------------------------------------
# EXECUÇÃO — teste com o exemplo exato do edital
# ------------------------------------------------------------
caminhos_lab = obter_caminhos_laboratorio()
artefatos = carregar_artefatos_producao(caminhos_lab['modelos'])
print("✅ Artefatos de produção carregados")

exemplo_edital = {
    "renda_mensal": 4500,
    "nivel_endividamento": 25,
    "frequencia_poupanca": "Media",
    "transacoes": [
        {"descricao": "Supermercado", "valor": 420},
        {"descricao": "Combustivel", "valor": 300},
        {"descricao": "Streaming", "valor": 40},
    ]
}

resultado = analisar_financas(exemplo_edital, artefatos)
print("\n📤 Resultado do pipeline (comparar com o exemplo do edital):")
print(json.dumps(resultado, indent=2, ensure_ascii=False))

Mounted at /content/drive
✅ Artefatos de produção carregados

📤 Resultado do pipeline (comparar com o exemplo do edital):
{
  "perfil_financeiro": "Saudavel",
  "probabilidade": 0.87,
  "resumo_gastos": {
    "Alimentacao": 420.0,
    "Transporte": 300.0,
    "Lazer": 40.0
  },
  "recomendacoes": [
    "Manter o padrão atual de organização financeira",
    "Considerar investir o excedente mensal para objetivos de longo prazo"
  ]
}
